# Stage 8B-1 — Stochastic Channel Realization Parity

Bu aşamada MATLAB RNG ile PyTorch RNG'yi eşitlemeye çalışmıyoruz.

MATLAB, `generate_channel_train` içinde kullandığı **gerçek random primitives**'leri export eder:

\[
XPR,\ ASA,\ ZSA,\ ASD,\ ZSD,
\]

cluster offset'leri ve her ray için \(2\times2\) polarization phase matrisi.

Sonra MATLAB aynı seed'i resetleyip gerçek `generate_channel_train` fonksiyonunu çağırır.

Python aynı primitives'i kullanarak:

\[
H_{BR}^{Python},H_{RU}^{Python}
\]

üretir ve MATLAB ile karşılaştırır.

Ayrıca küçük integration check:

\[
H_{BR},H_{RU}\to F\to W\to F_{\rm eff}\to Y
\]

de aynı dosyada kontrol edilir.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import sys, zipfile, shutil
import numpy as np
import pandas as pd
import torch

ROOT = Path('/content/drive/MyDrive/MyDrive/RIS')

required = [
    'ris_gpu_physics_stage1.py',
    'ris_gpu_channel_realizations_stage8b1.py',
]

for d in [ROOT,Path('/content')]:
    if str(d) not in sys.path:
        sys.path.insert(0,str(d))

missing = [
    name for name in required
    if not (ROOT/name).exists() and not (Path('/content')/name).exists()
]
assert not missing, "Eksik:\n" + "\n".join(missing)

from ris_gpu_channel_realizations_stage8b1 import compare_stage8b1_case

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print("Device:",device)
if torch.cuda.is_available():
    print("GPU:",torch.cuda.get_device_name(0))

## MATLAB golden export

MATLAB path'inde güncel:

```text
generate_channel_train.m
generate_cascaded_ch.m
generate_eff_ch.m
```

ve önceki environment fonksiyonları bulunurken:

```matlab
export_stage8b1_channel_primitives_suite
```

çalıştır.

Çıktı:

```text
stage8b1_channel_golden.zip
```

Bunu `/content` altına yükle.

In [ ]:
ZIP = Path('/content/stage8b1_channel_golden.zip')
assert ZIP.exists(), "stage8b1_channel_golden.zip /content altında yok."

EXTRACT = Path('/content/stage8b1_channel_extract')
if EXTRACT.exists():
    shutil.rmtree(EXTRACT)
EXTRACT.mkdir(parents=True)

with zipfile.ZipFile(ZIP,'r') as zf:
    zf.extractall(EXTRACT)

mans = list(EXTRACT.rglob('manifest.csv'))
assert len(mans)==1,mans
SUITE = mans[0].parent
manifest = pd.read_csv(mans[0])

display(manifest)
assert len(manifest)==8

In [ ]:
# DOUBLE / COMPLEX128 FIXED-PRIMITIVE PARITY

rows=[]

for _,meta in manifest.iterrows():
    print(
        f"{meta['caseName']} | "
        f"nT={meta['nT']} nR={meta['nR']} nRIS={meta['nRIS']}"
    )

    r=compare_stage8b1_case(
        str(SUITE/str(meta['file'])),
        device=device,
        parity=True,
    )
    rows.append(r)

df64=pd.DataFrame(rows)

show=[
    'scenario','nT','nR','nRIS','N',
    'BR_H_relFro','RU_H_relFro',
    'F_relFro','Feff_relFro','Y_relFro',
]
display(df64[show])

rel_cols=[c for c in df64.columns if c.endswith('_relFro')]
worst=float(df64[rel_cols].to_numpy().max())
where=np.unravel_index(
    np.argmax(df64[rel_cols].to_numpy()),
    df64[rel_cols].shape
)

print(
    "Worst double:",
    df64.iloc[where[0]]['scenario'],
    rel_cols[where[1]],
    worst
)

assert worst < 1e-10, (
    f"Stage 8B-1 double parity failed: {worst:.3e}"
)

print("PASS: Stage 8B-1 stochastic fixed-primitives parity")

In [ ]:
# FLOAT32 / COMPLEX64 PRODUCTION SANITY

rows=[]

for _,meta in manifest.iterrows():
    r=compare_stage8b1_case(
        str(SUITE/str(meta['file'])),
        device=device,
        parity=False,
    )
    rows.append(r)

df32=pd.DataFrame(rows)
display(df32[show])

rel_cols=[c for c in df32.columns if c.endswith('_relFro')]
worst32=float(df32[rel_cols].to_numpy().max())

print("Worst float32:",worst32)

assert worst32 < 1e-4, (
    f"Stage 8B-1 float32 sanity failed: {worst32:.3e}"
)

print("PASS: Stage 8B-1 float32 stochastic sanity")

In [ ]:
# Save summaries.
OUT64=Path('/content/stage8b1_double_summary.csv')
OUT32=Path('/content/stage8b1_float32_summary.csv')

df64.to_csv(OUT64,index=False)
df32.to_csv(OUT32,index=False)

print(OUT64)
print(OUT32)

## Stage 8B-1 geçerse

Şu stochastic zincir deterministic olarak kapanmış olur:

\[
\boxed{
\text{fixed random primitives}
\to H_{BR},H_{RU}
\to F
\to W
\to F_{\rm eff}
\to Y
}
\]

Sonraki aşamada MATLAB primitives yerine GPU-native random primitive generation
ekleyip distribution/statistical parity ve milyon-realization throughput
benchmark'ına geçeceğiz.